# Titanic: Data Prep

Turns the raw CSVs into the tensor files the modeling notebooks load. The reasoning behind each feature is in `00_eda`.

Re-run this notebook whenever the features change.

In [1]:
import numpy as np
import pandas as pd

import torch

## Load Datasets

In [2]:
df = pd.read_csv('../data/train.csv')
df_test = pd.read_csv('../data/test.csv')

df.shape, df_test.shape

((891, 12), (418, 11))

## Preprocessing

`add_features` holds the row-wise steps that need no statistics. `Preprocessor.fit` learns every statistic (fill values, means/stds, one-hot columns) from the data it is fitted on, and `transform` applies them unchanged to any other split.

In [3]:
# Spelling variants fold into their common title; everything else that is rare
# (one-off honorifics like Jonkheer, or test-only ones like Dona) becomes 'Rare'.
TITLE_ALIASES = {'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'}
COMMON_TITLES = ['Mr', 'Mrs', 'Miss', 'Master']

# Passengers per ticket, counted over train and test together (no labels involved).
# Fare is the price of the whole ticket, so it is split by this count.
TICKET_COUNTS = pd.concat([df, df_test])['Ticket'].value_counts()


def add_features(df):
    df = df.copy()

    title = df['Name'].str.extract(r',\s*([^.]*)\.')[0].str.strip().replace(TITLE_ALIASES)
    df['Title'] = title.where(title.isin(COMMON_TITLES), 'Rare')

    # Women in 1st/2nd class almost all survived, in 3rd only half; a linear model needs this spelled out.
    df['Sex_Pclass'] = df['Sex'] + '_' + df['Pclass'].astype(str)

    df['AgeMissing'] = df['Age'].isna().astype(int)
    df['HasCabin'] = df['Cabin'].notna().astype(int)

    df['TicketCount'] = df['Ticket'].map(TICKET_COUNTS)
    df['LogFarePP'] = np.log1p(df['Fare'] / df['TicketCount'])

    # Survival peaks for families of 2-4, so size is binned rather than kept linear.
    family_size = df['SibSp'] + df['Parch'] + 1
    df['FamilyGroup'] = pd.cut(family_size, bins=[0, 1, 4, np.inf], labels=['alone', 'small', 'large']).astype(str)

    return df.drop(
        columns=['PassengerId', 'Name', 'Ticket', 'Cabin', 'Survived', 'SibSp', 'Parch', 'Fare', 'Embarked'],
        errors='ignore',
    )


def woman_child_groups(df, X):
    """Family key for each woman or boy; NaN for men, who are left out of group survival."""
    is_woman_or_boy = (X['Sex'] == 'female') | (X['Title'] == 'Master')
    surname = df['Name'].str.split(',').str[0]

    # Relatives usually shared one ticket, so the fare separates unrelated people with the same surname.
    return (surname + '_' + df['Fare'].round(1).astype(str)).where(is_woman_or_boy)

In [4]:
class Preprocessor:
    def fit(self, df):
        X = add_features(df)

        # Survivors and known members per group, plus each fitted passenger's own label so
        # transform can leave it out.
        known = df['Survived'].groupby(woman_child_groups(df, X))
        self.group_survived = known.sum()
        self.group_known = known.count()
        self.own_label = df.set_index('PassengerId')['Survived']
        X = self._add_group_survival(df, X)

        # Title separates boys (Master) from men (Mr), so it fills Age far better than one global mean.
        self.age_by_title = X.groupby('Title')['Age'].median()
        self.fare_median = X['LogFarePP'].median()
        X = self._impute(X)

        self.numeric_cols = X.select_dtypes(include='number').columns.tolist()
        self.categorical_cols = X.select_dtypes(include=['object', 'string']).columns.tolist()

        self.means = X[self.numeric_cols].mean()
        self.stds = X[self.numeric_cols].std()

        self.columns = pd.get_dummies(X, columns=self.categorical_cols).columns
        
        return self

    def transform(self, df):
        X = self._impute(self._add_group_survival(df, add_features(df)))

        X[self.numeric_cols] = (X[self.numeric_cols] - self.means) / self.stds

        X = pd.get_dummies(X, columns=self.categorical_cols, dtype=int)
        
        return X.reindex(columns=self.columns, fill_value=0)

    def _add_group_survival(self, df, X):
        groups = woman_child_groups(df, X)

        # NaN for rows outside the fitted split (val/test), which have no label to remove.
        own = df['PassengerId'].map(self.own_label)
        survived = groups.map(self.group_survived) - own.fillna(0)
        known = groups.map(self.group_known) - own.notna()

        X['GroupDied'] = ((known > 0) & (survived == 0)).astype(int)
        X['GroupLived'] = ((known > 0) & (survived == known)).astype(int)

        return X

    def _impute(self, X):
        X['Age'] = X['Age'].fillna(X['Title'].map(self.age_by_title))

        return X.fillna({'LogFarePP': self.fare_median})

In [5]:
def to_tensor(data):
    return torch.tensor(data.to_numpy(), dtype=torch.float32)


def to_labels(df):
    return torch.tensor(df['Survived'].to_numpy(), dtype=torch.int64)

### Engineered Features

A quick check that every feature has the expected range and no NaNs before imputation (only `Age` should have gaps).

In [6]:
features = add_features(df)

features.describe()

,Pclass,Age,AgeMissing,HasCabin,TicketCount,LogFarePP
count,891.000000,714.000000,891.000000,891.000000,891.000000,891.000000
mean,2.308642,29.699118,0.198653,0.228956,2.121212,2.489616
std,0.836071,14.526497,0.399210,0.420397,1.796660,0.692634
min,1.000000,0.420000,0.000000,0.000000,1.000000,0.000000
25%,2.000000,20.125000,0.000000,0.000000,1.000000,2.139572
50%,3.000000,28.000000,0.000000,0.000000,1.000000,2.202765
75%,3.000000,38.000000,0.000000,0.000000,3.000000,2.696312
max,3.000000,80.000000,1.000000,1.000000,11.000000,4.860450


## Train/Val Split

Statistics are fitted on the 80% train split only and applied to both splits.

In [7]:
df_val = df.groupby('Survived').sample(frac=0.2, random_state=42)
df_train = df.drop(index=df_val.index)

len(df_train), len(df_val)

(713, 178)

In [8]:
split_prep = Preprocessor().fit(df_train)

df_train_X = split_prep.transform(df_train)
df_val_X = split_prep.transform(df_val)

df_train_X.shape, df_val_X.shape

((713, 24), (178, 24))

In [9]:
df_train_X[split_prep.numeric_cols].describe()

,Pclass,Age,AgeMissing,HasCabin,TicketCount,LogFarePP,GroupDied,GroupLived
count,7.130000e+02,7.130000e+02,7.130000e+02,7.130000e+02,7.130000e+02,7.130000e+02,7.130000e+02,7.130000e+02
mean,1.918366e-16,-1.494830e-17,2.615953e-17,-6.975875e-17,-7.723291e-17,3.487938e-17,-4.484491e-17,-6.975875e-17
std,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00
min,-1.571832e+00,-2.173599e+00,-4.763149e-01,-5.396829e-01,-6.386023e-01,-3.607150e+00,-2.337883e-01,-2.945647e-01
25%,-3.680553e-01,-6.068368e-01,-4.763149e-01,-5.396829e-01,-6.386023e-01,-4.884848e-01,-2.337883e-01,-2.945647e-01
50%,8.357219e-01,4.027114e-02,-4.763149e-01,-5.396829e-01,-6.386023e-01,-4.195853e-01,-2.337883e-01,-2.945647e-01
75%,8.357219e-01,4.589880e-01,-4.763149e-01,-5.396829e-01,4.815046e-01,2.625422e-01,-2.337883e-01,-2.945647e-01
max,8.357219e-01,3.428071e+00,2.096507e+00,1.850341e+00,4.961932e+00,3.426283e+00,4.271375e+00,3.390078e+00


In [10]:
torch.save({
    'X_train': to_tensor(df_train_X),
    'Y_train': to_labels(df_train),
    'X_val': to_tensor(df_val_X),
    'Y_val': to_labels(df_val),
    'columns': df_train_X.columns.tolist(),
}, '../data/train_val.pt')

## Full Train and Test

Statistics are fitted on all of `train.csv` and applied to `test.csv`, so a model trained on `train_full.pt` matches `test.pt`.

In [11]:
full_prep = Preprocessor().fit(df)

df_full_X = full_prep.transform(df)
df_test_X = full_prep.transform(df_test)

df_full_X.shape, df_test_X.shape

((891, 24), (418, 24))

In [12]:
torch.save({
    'X': to_tensor(df_full_X),
    'Y': to_labels(df),
    'columns': df_full_X.columns.tolist(),
}, '../data/train_full.pt')

torch.save({
    'X': to_tensor(df_test_X),
    'PassengerId': torch.tensor(df_test['PassengerId'].to_numpy()),
    'columns': df_test_X.columns.tolist(),
}, '../data/test.pt')

## Check Saved Tensors

In [13]:
for name in ('train_val', 'train_full', 'test'):
    data = torch.load(f'../data/{name}.pt')
    shapes = {k: tuple(v.shape) if torch.is_tensor(v) else len(v) for k, v in data.items()}
    nans = sum(v.isnan().sum().item() for v in data.values() if torch.is_tensor(v) and v.is_floating_point())
    print(name, shapes, 'NaNs:', nans)

train_val {'X_train': (713, 24), 'Y_train': (713,), 'X_val': (178, 24), 'Y_val': (178,), 'columns': 24} NaNs: 0
train_full {'X': (891, 24), 'Y': (891,), 'columns': 24} NaNs: 0
test {'X': (418, 24), 'PassengerId': (418,), 'columns': 24} NaNs: 0
